# Testing Wav2Vec2Phoneme on audios from OcOr (Corpus of Occitan Oral Narratives)

## 1. Imports

In [4]:
import os
from pathlib import Path

In [ ]:
from models.w2v2p import load_model, predict_simple
from data.get_data import get_phonemes_from_json
from data.form_dataset import make_dataset
from data.get_closest import closest
from data.cleaning import clean_list
from evaluation.metrics import get_per_nltk
from evaluation.pipeline import evaluate_dataset
from evaluation.alignment import alignement_ajuste

## 2. Load paths

In [5]:
base_path = Path("..")
ocor_path = base_path / "data/OOT"

## 3. Load model

In [4]:
processor, model = load_model("facebook/wav2vec2-xlsr-53-espeak-cv-ft")

Loading weights: 100%|██████████| 424/424 [00:00<00:00, 1196.21it/s, Materializing param=wav2vec2.masked_spec_embed]                                            


## 4. Load data

In [5]:
contes = {
    'taupa_grapaud':["part1", "part2", "part3"],
    'nau_lops': ["part1", "part2", "part3", "part4"],
}

In [6]:
sentences = []
for name, part in contes.items():
    for p in part:
        sent_path = ocor_path / f"{name}/{name}_{p}.txt"
        with open(sent_path, "r", encoding="utf-8") as file:
            sentences.append(file.read())

In [7]:
y_test = get_phonemes_from_json(base_path/"scripts_propres/phonemes_ocor.json")
y_test_taupa = get_phonemes_from_json(base_path/'scripts_propres/phonemes_taupa.json')
y_test_lops = get_phonemes_from_json(base_path/'scripts_propres/phonemes_nau_lops.json')

y_test_gascon = [t[1] for t in y_test]
y_test_taupa_gascon = [t[1] for t in y_test_taupa]
y_test_lops_gascon = [t[1] for t in y_test_lops]

In [8]:
X_test = []
X_test_taupa = []
X_test_lops =  []

for name, part in contes.items():
    audio_path = ocor_path/name
    for p in part:
        X_test.append(os.path.join(audio_path, f"{name}_{p}.mp3"))
    if name == "taupa_grapaud":
        for p in part:
            X_test_taupa.append(os.path.join(audio_path, f"{name}_{p}.mp3"))
    elif name == "nau_lops":
        for p in part:
            X_test_lops.append(os.path.join(audio_path, f"{name}_{p}.mp3"))

## 5. Build dataset

In [9]:
test_dataset = make_dataset(X_test, y_test)
taupa_dataset = make_dataset(X_test_taupa, y_test_taupa)
lops_dataset = make_dataset(X_test_lops, y_test_lops)

## 6. Run inference
- Tout le corpus
- La taupa e lo grapaud
- El conte dels nau lops


In [10]:
per_total, aligned_per_total = evaluate_dataset(
    test_dataset,
    X_test,
    y_test,
    model,
    processor,
    predict_simple,
    clean_list,
    closest,
    get_per_nltk,
    alignement_ajuste,
)

per_taupa, aligned_per_taupa = evaluate_dataset(
    taupa_dataset,
    X_test_taupa,
    y_test_taupa,
    model,
    processor,
    predict_simple,
    clean_list,
    closest,
    get_per_nltk,
    alignement_ajuste,
)

per_lops, aligned_per_lops = evaluate_dataset(
    lops_dataset,
    X_test_lops,
    y_test_lops,
    model,
    processor,
    predict_simple,
    clean_list,
    closest,
    get_per_nltk,
    alignement_ajuste,
)

100%|██████████| 1062/1062 [00:09<00:00, 117.88it/s]


In [11]:
print(f"PER sur tout le corpus : {per_total} | PER aligné sur tout le corpus : {aligned_per_total}")
print(f"PER La taupa e lo grapaud : {per_taupa} | PER aligné La taupa e lo grapaud : {aligned_per_taupa}")
print(f"PER El conte dels nau lops : {per_lops} | PER aligné El conte dels nau lops : {aligned_per_lops}")

PER sur tout le corpus : 46.24% | PER aligné sur tout le corpus : 47.35%
PER La taupa e lo grapaud : 38.84% | PER aligné La taupa e lo grapaud : 37.86%
PER El conte dels nau lops : 51.79% | PER aligné El conte dels nau lops : 50.98%
